# ML-04 · Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Huzaifah1805/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

**Lane:** Core Lane 2 — Content Refresh / Content Opportunity Scoring  
**Intern:** Huzaifah  
**Mid-panel month used:** `month = 2026-03`  
**Warehouse source:** `FlyRank/internship-warehouse` (Hugging Face, gated, via DuckDB)  
**Warning:** the `_sample` table is the *final* month (June 2026) — used only to test query mechanics, never for label development. All label and feature work is on `month=2026-03`.

## 0. Setup

In [ ]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

In [ ]:
import os, getpass
import duckdb
import pandas as pd
import numpy as np

# Colab: store token in Secrets panel as HF_TOKEN (never paste into a cell -- repo is public)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

print('Token loaded:', 'yes' if HF_TOKEN else 'NO -- set HF_TOKEN secret!')

In [ ]:
# Connect DuckDB to the Hugging Face warehouse
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients'      : f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content'      : f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily'       : f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d'   : f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Quick sanity check: row counts (reads Parquet metadata only -- fast)
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:25} {n:>12,} rows')

---
## 1) The Contract -- Five Plain-Words Answers

### 1. What does one row mean in my lane?
One row is **one pseudonymized content item (page) on one calendar day, for one client** in `fact_content_daily_performance` — grain: `(report_date, client_hash_id, content_hash_id)`. For the modelling frame I aggregate this into **one row per content item** across a 90-day feature window, so the decision grain is: *one content item at decision date 2026-03-31*.

### 2. Which table(s) will I use?
| Table | Role |
|---|---|
| `fact_content_daily_performance` | Time-series impressions, clicks, position, GA4 sessions — backbone of all features and the label |
| `dim_content` | Content metadata (word-count tier, content type, intent) — static context features |
| `dim_clients` | `gsc_data_start` / `ga4_data_start` — to filter clients with enough history |

I deliberately **do not** use `fact_content_query_90d` for the feature frame; its fixed 90-day window is anchored to the snapshot end (June 2026), so using it for a mid-panel decision point would leak future data.

### 3. Which time window?
```
Feature window : 90 days ending 2026-03-31  (Jan + Feb + Mar 2026)
Decision date  : 2026-03-31
Label window   : April 2026  (days +1...+30 after the decision date)
Dev month      : 2026-03  (mid-panel; sealed test month 2026-06 is untouched)
```
The feature window ends *before* the label window begins — no overlap, no leakage by construction.

### 4. What would I predict or rank (label / proxy)?
**Label:** *Did this content item lose >= 20 % of its GSC impressions in April 2026 compared to March 2026?*
```
is_declining = (impressions_april < 0.80 * impressions_march)
             AND (impressions_march >= 100)   -- noise filter
```
This is a **future-looking** label built only from observable GSC impressions — no product flags, no health scores.

### 5. One thing I deliberately exclude
**I exclude `ga4_data_available = FALSE` rows from all GA4-derived features** (sessions, engagement rate, scroll rate). These rows have zero-filled GA4 columns that look like 'no engagement' but actually mean 'no GA4 tracking yet for this client'. Treating them as real zeros would silently encode client identity into the features and introduce systematic bias. I filter to `ga4_data_available IS TRUE` whenever I compute any GA4 metric.

---
## 2) Prove Three Facts with Three Queries (month = 2026-03)

In [ ]:
# -- Query 1: Grain check -------------------------------------------------------
# Prove: one row really is one (report_date, client_hash_id, content_hash_id)
q1 = con.sql(f"""
    SELECT
        COUNT(*)                                                           AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))    AS distinct_keys,
        COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_triples
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

print('Query 1 -- Grain Check (month=2026-03):')
print(q1.to_string(index=False))
assert q1['duplicate_triples'].iloc[0] == 0, 'GRAIN VIOLATED -- duplicates found!'
print('\nVerdict: grain is (report_date, client_hash_id, content_hash_id) -- confirmed unique.')

In [ ]:
# -- Query 2: Row count and date span ------------------------------------------
# Prove: how many rows, clients, content items, and what is the actual date range
q2 = con.sql(f"""
    SELECT
        COUNT(*)                           AS total_rows,
        COUNT(DISTINCT client_hash_id)     AS distinct_clients,
        COUNT(DISTINCT content_hash_id)    AS distinct_content_items,
        MIN(report_date)                   AS date_min,
        MAX(report_date)                   AS date_max,
        MAX(report_date) - MIN(report_date) + 1 AS days_spanned
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

print('Query 2 -- Row Count & Date Span (month=2026-03):')
print(q2.to_string(index=False))
print('\nVerdict: slice profiled -- date range and row count confirmed above.')

In [ ]:
# -- Query 3: Availability -- IS TRUE filter ------------------------------------
# Prove: how many rows have ga4_data_available IS TRUE (usable GA4 data)
q3 = con.sql(f"""
    SELECT
        COUNT(*)                                                              AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE  THEN 1 ELSE 0 END)         AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END)         AS ga4_unavailable_rows,
        ROUND(
            100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
            / COUNT(*), 2
        )                                                                     AS ga4_available_pct
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

print('Query 3 -- Availability: ga4_data_available IS TRUE (month=2026-03):')
print(q3.to_string(index=False))
print('\nVerdict: GA4-derived features will be computed only on rows where ga4_data_available IS TRUE')
print('to avoid encoding client history depth as a spurious zero-signal into the feature matrix.')

---
## 3) Five Features + The Leakage Trap

### 3a -- Build the five-feature frame (honest version)

In [ ]:
# -- Build a 5-feature frame from the warehouse (Jan-Apr 2026) -----------------
# Feature window: Jan-Mar 2026 (90 days ending 2026-03-31)
# Label window:   Apr 2026    (30 days after decision date)
print('Building feature frame ... (reads Jan-Apr 2026 partitions; ~2-5 min on Colab)')

feature_frame = con.sql(f"""
WITH
feat AS (
    SELECT
        client_hash_id,
        content_hash_id,
        -- F1: Log-scaled total impressions in the 90-day feature window
        LN(SUM(gsc_impressions) + 1)                                                     AS f_log_impressions_90d,

        -- F2: Average GSC position over the feature window (lower = better ranking)
        AVG(NULLIF(gsc_avg_position, 0))                                                  AS f_avg_position_90d,

        -- F3: Aggregate CTR (clicks / impressions x 100) -- pre-decision aggregate
        ROUND(
            100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 4
        )                                                                                  AS f_ctr_90d,

        -- F4: Momentum ratio: March impressions / January impressions (both inside feature window)
        ROUND(
            COALESCE(SUM(CASE WHEN report_date >= DATE '2026-03-01' THEN gsc_impressions ELSE 0 END), 0) * 1.0
            / NULLIF(SUM(CASE WHEN report_date <  DATE '2026-02-01' THEN gsc_impressions ELSE 0 END), 0), 4
        )                                                                                  AS f_impression_momentum,

        -- F5: GA4 engagement rate -- only on rows where GA4 is actually tracking
        ROUND(
            100.0
            * SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE NULL END)
            / NULLIF(SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE NULL END), 0),
            4
        )                                                                                  AS f_engagement_rate_90d,

        -- March impressions -- needed for label denominator
        SUM(CASE WHEN report_date >= DATE '2026-03-01' THEN gsc_impressions ELSE 0 END)   AS imp_mar

    FROM {TABLES['fact_daily']}
    WHERE month IN ('2026-01', '2026-02', '2026-03')
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 100    -- noise filter: minimum volume
),
lbl AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_apr
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.f_log_impressions_90d,
    f.f_avg_position_90d,
    f.f_ctr_90d,
    f.f_impression_momentum,
    f.f_engagement_rate_90d,
    -- Label: >= 20% impression drop in April vs March, with volume guard
    CASE
        WHEN COALESCE(l.imp_apr, 0) < 0.80 * NULLIF(f.imp_mar, 0)
             AND f.imp_mar >= 100
        THEN 1
        ELSE 0
    END AS label_is_declining
FROM feat f
LEFT JOIN lbl l
    ON f.client_hash_id = l.client_hash_id
   AND f.content_hash_id = l.content_hash_id
""").df()

print(f'Feature frame: {len(feature_frame):,} content items x {len(feature_frame.columns)} columns')
print(f'Label balance: {feature_frame["label_is_declining"].mean():.1%} declining')
print()
print(feature_frame.head(5).to_string(index=False))

### Feature summary table

| # | Feature | What it measures | "Knowable at the decision moment because..." |
|---|---|---|---|
| F1 | `f_log_impressions_90d` | Log-scaled total GSC impressions Jan-Mar 2026 | Sum of a closed historical window (feature window ends 2026-03-31). GSC lag is 2-3 days, so all March data is settled at decision point. Log-scale suppresses heavy-tail skew. |
| F2 | `f_avg_position_90d` | Average GSC rank position over the feature window | Position data is reported alongside impressions in GSC at the same 2-3 day lag. A rising (worsening) average position is a leading signal of future decline. |
| F3 | `f_ctr_90d` | CTR = clicks / impressions x 100 over 90 days | Both clicks and impressions are 90-day historical aggregates from GSC — fully available at the decision date. CTR below the expected tier value signals title/meta mismatch. |
| F4 | `f_impression_momentum` | March impressions / January impressions | Both are inside the feature window (Jan and Mar 2026). The ratio captures short-term trajectory without touching the label window. A value < 0.8 already signals weakening visibility. |
| F5 | `f_engagement_rate_90d` | GA4 engaged sessions / total sessions x 100 | GA4 session data is available the next day; all feature-window rows are settled by 2026-03-31. Computed only on `ga4_data_available IS TRUE` rows to avoid encoding client history depth as a spurious zero. |

In [ ]:
# Feature summary statistics
feat_cols = ['f_log_impressions_90d','f_avg_position_90d','f_ctr_90d',
             'f_impression_momentum','f_engagement_rate_90d']
print('=== Feature summary statistics ===')
feature_frame[feat_cols].describe().round(3)

---
### 3b -- THE TRAP: Deliberate Label Leakage

> **The experiment:** add one future-window column (`imp_apr_raw`) as if it were a feature, watch the ROC AUC jump toward near-perfect, then **delete it** and keep only the honest score.
> This is the leakage lesson from notebook 02, performed on real warehouse data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# Step 1: pull April raw impressions (the LEAKED column)
april_raw = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_apr_raw
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
""").df()

df_leaked = feature_frame.merge(april_raw, on=['client_hash_id','content_hash_id'], how='left')
df_leaked['imp_apr_raw'] = df_leaked['imp_apr_raw'].fillna(0)
print(f'Leaked dataframe: {len(df_leaked):,} rows, columns: {df_leaked.columns.tolist()}')

In [ ]:
def quick_rf_score(frame, features, label='label_is_declining', seed=42):
    """Fit a quick Random Forest and return (ROC AUC, Avg Precision)."""
    clean = frame.dropna(subset=features)
    X, y = clean[features].values, clean[label].values
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=seed, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    prob = clf.predict_proba(X_te)[:, 1]
    return roc_auc_score(y_te, prob), average_precision_score(y_te, prob)


honest_feats = ['f_log_impressions_90d','f_avg_position_90d','f_ctr_90d',
                'f_impression_momentum','f_engagement_rate_90d']
leaked_feats = honest_feats + ['imp_apr_raw']

# Score the HONEST model
auc_honest, ap_honest = quick_rf_score(feature_frame, honest_feats)

# Score the LEAKED model
auc_leaked, ap_leaked = quick_rf_score(df_leaked, leaked_feats)

print('=== THE LEAKAGE TRAP ===')
print(f'HONEST model  -- ROC AUC: {auc_honest:.4f}   Avg Precision: {ap_honest:.4f}')
print(f'LEAKED model  -- ROC AUC: {auc_leaked:.4f}   Avg Precision: {ap_leaked:.4f}  <-- suspiciously high!')
print(f'\nLift from leaked feature: ROC AUC {auc_leaked-auc_honest:+.4f},  Avg Precision {ap_leaked-ap_honest:+.4f}')
print()
print('WHY it leaks: imp_apr_raw IS the numerator of the label.')
print('  A page with low April impressions almost guarantees label_is_declining = 1.')
print('  In production this column does not exist at decision time -- the model silently fails.')

In [ ]:
# Step 2: DELETE the leaked column -- discard df_leaked entirely
del df_leaked

print('Leaked column removed. Clean frame columns:')
print(feature_frame.columns.tolist())
print()
print(f'Honest ROC AUC to beat in Week 4: {auc_honest:.4f}')
print(f'Honest Avg Precision to beat:     {ap_honest:.4f}')
print()
print('Conclusion: the leaked model looked strong because it learned the label from itself.')
print('The honest score is the only number worth reporting.')

---
## 4) One Named Limitation of My Slice

**Limitation: unbalanced panel history -- not all clients have a full 90-day feature window before 2026-03-31.**

The warehouse is an **unbalanced panel**: `dim_clients.gsc_data_start` varies per client. Clients that joined the platform in late 2025 may not have full January-March 2026 history, meaning:
- `f_log_impressions_90d` is understated (fewer days in window)
- `f_impression_momentum` is noisier (smaller January baseline)
- The label may be undefined for clients with no April 2026 data

**Consequence:** The feature frame under-represents newer clients. A model trained on this frame may generalise poorly to recently-onboarded clients -- exactly the clients most likely to appear in future live deployments.

**Mitigation:** Filter to clients with `gsc_data_start <= DATE '2026-01-01'` before modelling. This reduces client coverage but guarantees every training example has a full 90-day feature window.

In [ ]:
# Quantify the limitation: clients with full vs partial 90-day window
panel_coverage = con.sql(f"""
    SELECT
        COUNT(*) AS total_clients,
        SUM(CASE WHEN gsc_data_start <= DATE '2026-01-01' THEN 1 ELSE 0 END) AS clients_full_90d,
        SUM(CASE WHEN gsc_data_start >  DATE '2026-01-01' THEN 1 ELSE 0 END) AS clients_partial_window
    FROM {TABLES['dim_clients']}
    WHERE gsc_data_start IS NOT NULL
""").df()

print('=== Panel coverage -- full vs partial 90-day feature window ===')
print(panel_coverage.to_string(index=False))
print('\nThe limitation is real and quantified. Partial-window clients will be filtered')
print('before any modelling to ensure training data quality.')

---
## 5) Self-Check

| Check | Status |
|---|---|
| Five plain-words contract answers written | ✅ Section 1 |
| Three verification queries executed with output visible | ✅ Section 2 |
| Availability checked with `ga4_data_available IS TRUE` | ✅ Query 3 |
| Five-feature frame built from mid-panel month (2026-03) | ✅ Section 3a |
| Every feature has a "knowable at decision moment because..." line | ✅ Feature table in 3a |
| Deliberate leak experiment shown: column added, score jumped, column deleted | ✅ Section 3b |
| Honest score kept and named as the number to beat in Week 4 | ✅ Section 3b |
| One named limitation of the slice, with quantification | ✅ Section 4 |
| Final month (2026-06) NOT used for label development | ✅ Throughout |
| No HF token pasted into code cells | ✅ Loaded from Colab Secrets / env variable |
| No raw client names, domains, URLs, or queries in outputs | ✅ Pseudonymized hash IDs only |